In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import pickle

# fast parquet
try:
    import fastparquet
except:
    ! pip install fastparquet
    import fastparquet

# binning
try:
    from optbinning import OptimalBinning
except:
    ! pip install optbinning
    from optbinning import OptimalBinning

#### Functions

In [ ]:
def bin_features(list_cols, X, y):
    dict_bins = {}
    list_cols_scorecard = []
    for col in tqdm(list_cols):        
        # init
        cls_binning = OptimalBinning(
            name=col,
            dtype='numerical',
            solver='cp',
            monotonic_trend=None,
            prebinning_method='cart',
            user_splits=None,
            user_splits_fixed=None,
        )
        # fit
        cls_binning.fit(
            X[col],
            y,
        )
        # assign
        dict_bins[col] = cls_binning
    # return
    return dict_bins

#### Constants

In [ ]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'

str_target = 'has_inst_tag'

list_str_inst = [
    # from ben: 2025-02-21
    'CURRENT',
    'SELF',
    # key words
    'CHIME-STRIDE',
    'CHIMEFINAL',
    # from dustin: 2025-02-24
    'SELF FIN',
    'SELF/LEAD',
    'SELFINC/LEAD',
    'SBNASELFLNDR',
    'SBNA SELF',
    'CHIME',
    'CLEO',
    'CLEO AI',
    'VARO',
    'ATLAS',
    'ATLCAPBKSELF',
    'POSSIBLE',
    'POSSIBLE FIN',
    'KIKOFF',
    'SUPER.COM',
    'STEP',
    'STEP MOBILE',
    'BRIGHT',
    'BRIGHT BLDR',
    'FIG TECH INC',
    'SELF/RENT',
    'SELFBILLSE',
    'PROGRESSRES',
    'FLEX',
    'FLEXFINANCE',
]

# rm no variance
list_cols_novariance = [
    'sr04s__tu',
    'sr60s__tu',
    'sd04s__tu',
    'sd05s__tu',
    'sd64s__tu',
    'sd71s__tu',
    'sd72s__tu',
    'sd85s__tu',
    'sd86s__tu',
    'sd87s__tu',
    'sd50s__tu',
    'linkc012__tu',
    'linkc010__tu',
    'linkc051__tu',
    'fltinsuredlifepremium__app',
    'fltinsuredunemploymentamount__app',
    'fltinsuredunemploymentpremium__app',
    'fltaddfee__app',
    'fltinsureddisabilitypremium__app',
    'sp04s__tu',
    'sp06s__tu',
    'sp07s__tu',
    'sp60s__tu',
    'sg05s__tu',
    'sp71s__tu',
    'hr29s__tu',
    'lienjudgmentforeclosurecount__ln',
    'purchaseactivityindex__ln',
    'inputprovidedfirstname__ln',
    'inputprovidedlastname__ln',
    'inputprovideddateofbirth__ln',
    'addrcurrentcorrectional__ln',
    'addrpreviouscorrectional__ln',
    's208s__tu'
]

# force features
list_cols_force = [
    'ENG-franchise',
    'ENG-loan_to_value',
    'ENG-bk',
    'ENG-wtd_avg',
    'fltgrossmonthly__income_sum',
    'miles_odometer__app',
    'ENG-vehicle_age',
    'g232s__tu',
    'rp01s__tu',
    'g106s__tu',
    'au20s__tu'
    
]
# odometer
# ltv
# bk
# franchise
# wtd avg
# veh age

# rm cols with all nans
list_cols_nan = [
    'linkb006__tu',
    'linkb007__tu',
     'linkc014__tu',
     'linkc015__tu',
     'linkc016__tu',
     'linkc022__tu',
     'linkc023__tu',
     'approvaldate__app',
     'fundeddate__app',
     'dtmapproved__app',
     'dtmdeclined__app',
     'defaultdate__app',
     'chargeoffdate__app',
     'defaultamount__app',
     'chargeoffamount__app',
     'bittrade__app',
     'purchaseactivitycount__ln',
     'purchaseactivitydollartotal__ln',
     'attribute_index__ln',
     'bkc203__tu',
     'bkc204__tu',
     'bkc205__tu',
     'bkc222__tu',
     'bkc223__tu',
     'bkc224__tu',
     'bkc225__tu',
     'bkc202__tu',
     'bkc201__tu',
     'bkc231__tu',
     'bkc232__tu',
     'bkc234__tu',
     'bkc235__tu',
     'bkc252__tu',
     'bkc253__tu',
     'bkc254__tu',
     'bkc255__tu',
     'bkc233__tu',
]

# cols with IDs to remove
list_ids = [
    'bigdealertypeid__app',
    'bigstatusid__app',
    'bigdealerid__app',
    'biglnriskviewattributesv5id__ln',
    'biglnriskviewscoreid__ln',
    'inputprovidedlexid__ln',    
]

# dict monotone constraints
dict_monotone_constraints = None

int_n_feats = 200

#### Output directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Load in data

In [ ]:
str_filename = 'df.gzip'
str_uri = f's3://20241112-simple-model-test/08_prep_data/{str_filename}'
df = pd.read_parquet(
    str_uri,
)
# sort
df.sort_values(by='request_datetime', ascending=True, inplace=True)
df

#### Create target

In [ ]:
df['list_institutions'] = df['str_institution__tu_pmthx'].apply(
    lambda x: eval(x.replace('nan','None')),
)
df['list_institutions'] = df['list_institutions'].apply(
    lambda x: [] if x is None else x,
)

list_str_col_new = []
for str_inst in tqdm(list_str_inst):
    str_col_new = f'{str_inst}_tag'
    df[str_col_new] = df['list_institutions'].apply(
        lambda x: 1 if str_inst in x else 0,
    )
    list_str_col_new.append(str_col_new)

df['sum'] = df[list_str_col_new].sum(axis=1)
df['has_inst_tag'] = df['sum'].apply(
    lambda x: 1 if x > 0 else 0,
)
flt_mn = df['has_inst_tag'].mean()
print(f'Proportion has tag: {flt_mn:0.4f}')
# show
df

#### Load in df feature importance

In [ ]:
# save
str_filename = 'df_feat_imp.csv'
str_local_path = f'../04_credit_builder_feats/{str_dirname_output}/{str_filename}'
df_tmp = pd.read_csv(str_local_path)
df_tmp.columns = ['feature', 'No', 'Yes', 'description']

# make rank
df_tmp['rank'] = range(0, df_tmp.shape[0])

# dict
dict_rank = dict(zip(df_tmp['feature'], df_tmp['rank']))

# list_cols 
list_cols_use = df_tmp['feature'].tolist()

# # rm non-numeric
list_cols_use = [col for col in list_cols_use if df[col].dtype in ['int64','float64']]

# # rm id cols
list_cols_use = [i for i in list_cols_use if i not in list_ids]

# remove cols with no variance
list_cols_use = [i for i in list_cols_use if i not in list_cols_novariance]

# remove cols with all nans
list_cols_use = [i for i in list_cols_use if i not in list_cols_nan]

# use top n
list_cols_use = list_cols_use[:int_n_feats]

# add forced features and remove dupes
list_cols_use = list(set(list_cols_use + list_cols_force))

# subset df
df_tmp = df[list_cols_use]

# show 
df_tmp

In [ ]:
[col for col in list(df_tmp.columns) if 'ENG' in col]

#### Get bins

In [ ]:
dict_bins = bin_features(
    list_cols=list_cols_use,
    X=df_tmp,
    y=df['has_inst_tag'],
)
# pickle dict_bins
str_filename = 'dict_bins.pkl'
str_local_path = f'./output/{str_filename}'
pickle.dump(dict_bins, open(str_local_path, 'wb'))

#### Show bins in order of importance

In [ ]:
# only get the top n features from feature importance because we don't have room

# show binning table
list_df = []
for key, val in tqdm(dict_bins.items()):
    if key in list_cols_use:
        df_bins = val.binning_table.build()
        list_df.append(df_bins)
        df_bins['Feature'] = key
    else:
        pass

# concat
df = pd.concat(list_df)

# get the aa dict
list_cols = [
    'feature_name',
    'Description',
]
str_filename = 'data_dictionary.csv'
str_local_path = f'./{str_filename}'
df_dd = pd.read_csv(str_local_path, usecols=list_cols)

# rename
dict_rename = {
    'feature_name': 'Feature',
    'Description': 'Description',
}
df_dd.rename(columns=dict_rename, inplace=True)

# join
df = pd.merge(
    left=df,
    right=df_dd,
    on='Feature',
    how='left',
)

# reorder
list_cols = [
    'Feature',
    'Bin',
    'Count',
    'Count (%)',
    'Non-event',
    'Event',
    'Event rate',
    'WoE',
    'IV',
    'JS',
    'Description',
]
df = df[list_cols].copy()

# rank
df['rank'] = df['Feature'].map(dict_rank)

# sort
df.sort_values(by='rank', ascending=True, inplace=True)

# save
str_filename = f'df_bins_top_{int_n_feats}.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

# show
df